# Contrastive Language-Image Pre-training（CLIP）模型

> **论文**：*Learning Transferable Visual Models From Natural Language Supervision*（OpenAI, 2021）  
> **核心思想**：以 **对比学习** 为目标，在 4 亿图文对上联合训练图像编码器（ViT）和文本编码器（Transformer），使同一语义的图文特征在嵌入空间中彼此靠近，不同语义的远离。  
> **零样本能力**：训练后无需微调，仅通过文本 Prompt 即可完成图像分类、检索等任务。

## 一、环境依赖检查 & CLIP 官方 API 快速推理

### 1.1 环境检查

确认 torch、torchvision、transformers 版本，以及 CUDA 可用性。

In [1]:
!pip list|grep torch

torch                                 2.6.0+cu124
torchao                               0.10.0
torchaudio                            2.6.0+cu124
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.21.0+cu124


In [4]:
!pip list|grep transformers

sentence-transformers                 4.1.0
transformers                          4.53.2


### 1.2 CLIP 官方 API 快速推理演示

加载 `openai/clip-vit-base-patch32`，对一张图像和三段候选文本做零样本分类推理，
输出各文本与图像的匹配概率分布。

In [ ]:
from transformers import CLIPProcessor, CLIPModel  # 从HuggingFace导入CLIP模型类和处理器类
from PIL import Image  # 导入Pillow库，用于加载本地图像文件
CACHE_DIR="./model_cache/CLIP"

# 从HuggingFace Hub加载预训练的CLIP模型（ViT-B/32版本：ViT backbone，patch_size=32）
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPModel 实例，含 text_model、vision_model、投影层、logit_scale 四个子模块
# 加载对应处理器：包含图像处理器（resize/归一化）和文本分词器（BPE tokenizer）
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPProcessor，内含 image_processor（归一化/resize）和 tokenizer（BPE分词）
print(model)  # 打印CLIP完整模型结构（含text_model、vision_model、投影层、logit_scale）
print('-'*100)  # 打印分隔线，分隔模型结构与处理器信息
print(processor)  # 打印处理器配置（图像归一化均值/方差和tokenizer词表大小等信息）

# 加载本地图像，供推理使用
image = Image.open('cat.png')  # 返回 PIL.Image 对象（RGB格式），后续由处理器 resize 至 224×224 并归一化

# 定义候选文本描述列表，用于零样本分类（Zero-Shot Classification）
texts = ["a photo of a cat", "a photo of a dog", "a photo of a girl"]  # 3条候选描述，类型 List[str]，模型计算每条与图像的余弦相似度

# 处理器同时预处理文本和图像：
# - text部分：BPE分词 → input_ids（token整数序列）+ attention_mask（有效位为1/padding位为0），形状均为 [3, max_len]
# - image部分：resize到224×224、归一化（均值/方差来自预训练设置）→ pixel_values，形状为 [1, 3, 224, 224]
# padding=True：将3段文本填充到相同长度（max_len为最长那段分词后的长度）
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)  # 返回 BatchFeature（类dict），键为 input_ids/attention_mask/pixel_values
print(inputs)  # 打印预处理后的输入字典（各张量键名和形状）

# 执行CLIP前向传播：文本和图像分别经过各自编码器，再经投影层对齐到同一嵌入空间
outputs = model(**inputs)  # 返回 CLIPOutput 对象，含以下关键属性：
# - image_embeds   : Tensor，形状 (1, 512)，图像经 ViT + 投影层后的 L2 归一化嵌入向量，1=图像数，512=嵌入维度
# - text_embeds    : Tensor，形状 (3, 512)，文本经 Transformer + 投影层后的 L2 归一化嵌入向量，3=文本数，512=嵌入维度
# - logits_per_image: Tensor，形状 (1, 3)，图像→文本方向的相似度得分矩阵
#     计算方式：logit_scale × image_embeds @ text_embeds.T，即 (1,512)@(512,3)×scale → (1,3)
#     含义：第[i,j]元素 = 第i张图像与第j段文本的余弦相似度×温度系数；对每行做 softmax 可得图像匹配各文本的概率分布
# - logits_per_text : Tensor，形状 (3, 1)，文本→图像方向的相似度得分矩阵，为 logits_per_image 的转置
#     含义：第[j,i]元素 = 第j段文本与第i张图像的余弦相似度×温度系数；对每行做 softmax 可得文本匹配各图像的概率分布
logits_per_image = outputs.logits_per_image  # 图像对每段文本的相似度得分，类型 Tensor，形状 (1, 3)：1张图像 × 3段文本，值越大越匹配
probs = logits_per_image.softmax(dim=1)  # 对3段文本的得分沿dim=1做softmax，得到概率分布，形状 (1, 3)，3个值之和严格为1

# 打印最终的概率分布：最高概率对应的文本为模型认为最匹配的描述
print("Probabilities:", probs)  # 打印形状 (1, 3) 的概率张量，每列对应一段候选文本的匹配概率

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((51

## 二、CLIP 官方接口深度解析

### 2.1 前向传播过程详细分解

逐步拆解 CLIP 前向传播的每个阶段，打印各中间张量的形状，理解从输入到输出的完整数据流：
- 文本编码 → 文本特征投影 → L2归一化
- 图像编码 → 图像特征投影 → L2归一化  
- 相似度矩阵计算 → softmax 概率分布 → 最优匹配

### 2.2 L2 归一化原理详解

**L2 归一化（L2 Normalization）** 是将一个向量除以其 **L2 范数（欧几里得范数）**，使得归一化后的向量长度（模）为 1，即将向量投影到单位超球面上。

**公式：**

$$\hat{x} = \frac{x}{\|x\|_2} = \frac{x}{\sqrt{\sum_{i} x_i^2}}$$

---

#### 在 CLIP 中的作用

1. **图像编码器** 输出图像特征向量 → L2 归一化 → 单位向量
2. **文本编码器** 输出文本特征向量 → L2 归一化 → 单位向量
3. 归一化后，两个向量的**点积**等价于**余弦相似度**：

$$\cos(\theta) = \hat{x}_{image} \cdot \hat{x}_{text}$$

---

#### 为什么要做 L2 归一化？

| 原因 | 说明 |
|------|------|
| 消除模长影响 | 只关注方向（语义），不受向量大小干扰 |
| 点积 = 余弦相似度 | 计算简单高效，便于对比学习 |
| 数值稳定 | 防止特征值过大导致梯度爆炸 |
| 对比损失需要 | InfoNCE Loss 依赖余弦相似度矩阵 |

---

#### PyTorch 实现

```python
import torch.nn.functional as F

# p=2 表示 L2 范数，dim=-1 表示沿最后一个维度（嵌入维度）归一化
# 输入形状：(batch_size, embed_dim)
# 输出形状：(batch_size, embed_dim)，每行向量的 L2 范数 = 1
features = F.normalize(features, p=2, dim=-1)
```

> **注意**：在 `CLIPModel` 的完整前向传播中，`outputs.text_embeds` 和 `outputs.image_embeds` 已经是 L2 归一化后的单位向量；而手动调用 `model.text_projection(...)` 时，需要自行做 L2 归一化。

In [4]:
from transformers import CLIPProcessor, CLIPModel  # 从HuggingFace导入CLIP模型类和处理器类
from PIL import Image  # 导入Pillow库，用于加载图像文件
import torch  # 导入PyTorch，提供张量操作和模型推理能力

# 从HuggingFace Hub加载预训练CLIP模型（ViT-B/32：图像编码器使用32×32 patch的ViT）
# 注意：CLIP 中的 ViT 与原始 ViT 的关键区别——去掉了分类头（MLP Head）
# 原始 ViT 完整流程：
#   图像 → patch embedding + position embedding
#         → 12层 Transformer（自注意力 + FFN）
#         → 取 [CLS] token 的隐藏状态
#         → LayerNorm → MLP Head(Linear) → 类别 logits（用于 ImageNet 等分类任务）
# CLIP ViT 完整流程：
#   图像 → patch embedding + position embedding
#         → pre_layrnorm（前置归一化，原始ViT无此步骤）
#         → 12层 Transformer（自注意力 + FFN）
#         → 取 [CLS] token 的隐藏状态
#         → post_layernorm → visual_projection(Linear 768→512) → 512维图像嵌入（用于图文相似度计算）
# 即原来的 MLP 分类头被替换为一个投影层，将视觉特征对齐到与文本嵌入相同的语义空间，而非输出类别概率
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPModel 实例，含 text_model(12层Transformer)、vision_model(ViT-B/32，无分类头)、投影层和 logit_scale
# 加载对应的CLIP处理器（负责图像预处理和文本BPE分词）
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",cache_dir=CACHE_DIR)  # 返回 CLIPProcessor，内含 image_processor（resize/归一化）和 tokenizer（BPE，词表大小49408）

print("CLIP Model Architecture:")  # 提示即将打印模型结构
print(model)  # 打印完整CLIP模型结构（含text_model、vision_model、投影层、logit_scale）
print('-'*100)  # 打印分隔线，分隔模型结构与其他信息
print('='*100)  # 打印双重分隔线，标记新的信息块

image = Image.open('cat.png')  # 加载本地图像文件，返回 PIL.Image 对象（RGB格式），供处理器预处理使用

# 定义三段候选文本，用于图文相似度零样本分类
texts = ["a photo of a cat", "a photo of a dog", "a photo of a girl"]  # 3条候选描述，类型 List[str]，每条将被BPE分词后与图像计算余弦相似度

# 使用处理器同时预处理文本（BPE分词）和图像（resize/归一化），padding=True使文本等长
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)  # 返回 BatchFeature（类dict），含 input_ids(3,max_len)、attention_mask(3,max_len)、pixel_values(1,3,224,224)
print("Processed Inputs:")  # 提示打印预处理后的输入信息
print(f"Input keys: {list(inputs.keys())}")  # 打印输入字典的所有键名，如 ['input_ids', 'attention_mask', 'pixel_values']
for key, value in inputs.items():  # 遍历每个输入张量，打印其形状或类型
    if isinstance(value, torch.Tensor):  # 如果是张量，打印形状
        print(f"  {key}: {value.shape}")  # 格式：键名: 张量形状，如 input_ids: torch.Size([3, 7])
    else:  # 如果不是张量（如列表），打印类型
        print(f"  {key}: {type(value)}")  # 打印非张量类型，如 <class 'list'>
print('-'*50)  # 打印分隔线，分隔输入信息与前向传播分析

model.eval()  # 将模型设为评估模式：关闭Dropout，固定BatchNorm统计量，影响推理结果确定性

with torch.no_grad():  # 推理阶段关闭梯度计算，节省显存和计算资源（无需反向传播）
    print("Forward Pass Analysis:")  # 提示开始逐步分析前向传播
    print('-'*50)  # 打印分隔线

    # ============ 阶段1：文本编码 ============
    print("1. Text Encoding:")  # 打印阶段标题
    text_inputs = {  # 构造仅含文本相关键的输入字典，供 text_model 单独调用
        'input_ids': inputs['input_ids'],          # 文本token ID，形状 (3, max_len)：3段文本，每段填充至最大长度
        'attention_mask': inputs['attention_mask']  # 注意力掩码，形状 (3, max_len)：有效token位置为1，padding位置为0
    }  # text_inputs 类型 dict，键为字符串，值为 Tensor
    print(f"   Text input_ids shape: {text_inputs['input_ids'].shape}")  # 打印文本token ID张量形状：(num_texts, max_seq_len)
    print(f"   Text attention_mask shape: {text_inputs['attention_mask'].shape}")  # 打印注意力掩码形状：(num_texts, max_seq_len)

    # 通过CLIP的text_model（12层因果Transformer）编码文本
    text_outputs = model.text_model(**text_inputs)  # 返回 BaseModelOutputWithPooling，含两个关键属性：
    # - last_hidden_state: Tensor，形状 (3, max_len, 512)
    #     3=文本数，max_len=最长序列的token数（含<SOS>/<EOS>/PAD），512=每个token位置的隐藏维度
    #     每个token都有独立的512维向量，例如 "a photo of a cat" → [<SOS>,a,photo,of,a,cat,<EOS>,...] 各自对应一个512维向量
    # - pooler_output: Tensor，形状 (3, 512)
    #     整段文本的句子级全局表示，通过取 last_hidden_state 中 [EOS] token 位置的向量得到
    #     即 last_hidden_state[:, eos_position, :] → (3, 512)
    #     选 [EOS] 的原因：CLIP 使用 GPT 风格的单向（因果）Transformer，[EOS] 能通过注意力看到前面所有token，天然聚合了全句语义
    # 通过文本投影头将 pooler_output 的512维向量映射到图文共享嵌入空间（512维，与图像特征维度对齐）
    text_embeds = model.text_projection(text_outputs.pooler_output)  # 对 [EOS] 位置特征做线性投影，返回 Tensor，形状 (3, 512)；经L2归一化后可与图像嵌入做余弦相似度

    # 打印文本编码器各阶段输出形状
    print(f"   Text model hidden states shape: {text_outputs.last_hidden_state.shape}")  # 形状 (3, max_len, 512)：3段文本，每位置512维隐藏状态
    print(f"   Text model pooler output shape: {text_outputs.pooler_output.shape}")  # 形状 (3, 512)：取每段文本[EOS]位置的特征作为句子表示
    print(f"   Text embeddings (after projection) shape: {text_embeds.shape}")  # 形状 (3, 512)：经线性投影映射到共享嵌入空间
    print()  # 空行分隔

    # ============ 阶段2：图像编码 ============
    print("2. Vision Encoding:")  # 打印阶段标题
    vision_inputs = {  # 构造仅含图像相关键的输入字典，供 vision_model 单独调用
        'pixel_values': inputs['pixel_values']  # 预处理后的图像张量，形状 (1, 3, 224, 224)：1张图，RGB三通道，224×224分辨率
    }  # vision_inputs 类型 dict
    print(f"   Vision pixel_values shape: {vision_inputs['pixel_values'].shape}")  # 打印图像张量形状：(batch_size, channels, height, width)

    # 通过CLIP的vision_model（ViT-B/32，12层Transformer，patch_size=32）编码图像
    vision_outputs = model.vision_model(**vision_inputs)  # 返回 BaseModelOutputWithPooling，含两个关键属性：
    # - last_hidden_state: Tensor，形状 (1, 50, 768)
    #     1=图像数，50=token数（1个[CLS]token + 49个patch token，49=7×7，来自224÷32=7），768=每个token的隐藏维度
    #     其中索引0位置为 [CLS] token，索引1~49为各 patch 的隐藏状态
    # - pooler_output: Tensor，形状 (1, 768)
    #     整张图像的全局表示，计算方式：post_layernorm( last_hidden_state[:, 0, :] )
    #     即取索引0的 [CLS] token 隐藏状态，再经 post_layernorm 归一化得到
    #     [CLS] token 在训练中通过注意力机制聚合了全部49个patch的信息，天然作为图像级全局特征
    #     注意与文本编码器的区别：文本用 [EOS] 位置（序列末尾），图像用 [CLS] 位置（序列开头，索引0）
    # 通过视觉投影头将 pooler_output 的768维向量映射到图文共享嵌入空间（512维）
    image_embeds = model.visual_projection(vision_outputs.pooler_output)  # 对 [CLS] 位置特征做线性投影（768→512），返回 Tensor，形状 (1, 512)；经L2归一化后可与文本嵌入做余弦相似度

    # 打印视觉编码器各阶段输出形状
    print(f"   Vision model hidden states shape: {vision_outputs.last_hidden_state.shape}")  # 形状 (1, 50, 768)：50=1([CLS])+49(7×7 patches)，每位置768维
    print(f"   Vision model pooler output shape: {vision_outputs.pooler_output.shape}")  # 形状 (1, 768)：取 [CLS] token 的特征作为图像全局表示
    print(f"   Image embeddings (after projection) shape: {image_embeds.shape}")  # 形状 (1, 512)：经线性投影从768维映射到512维共享嵌入空间
    print()  # 空行分隔

    # ============ 阶段3：完整前向传播（一次性执行所有步骤）============
    print("3. Complete Forward Pass:")  # 打印阶段标题
    outputs = model(**inputs)  # 一次性前向传播，自动完成编码、投影、L2归一化和相似度计算，返回 CLIPOutput

    print(f"   Final text features shape: {outputs.text_embeds.shape}")  # 最终文本特征形状 (3, 512)，已经过L2归一化（单位向量）
    print(f"   Final image features shape: {outputs.image_embeds.shape}")  # 最终图像特征形状 (1, 512)，已经过L2归一化（单位向量）
    print(f"   Logits per image shape: {outputs.logits_per_image.shape}")  # 图像对文本的得分矩阵，形状 (1, 3)：1张图像 × 3段文本
    print(f"   Logits per text shape: {outputs.logits_per_text.shape}")  # 文本对图像的得分矩阵，形状 (3, 1)：3段文本 × 1张图像
    print()  # 空行分隔

    # ============ 阶段4：相似度计算原理解析 ============
    print("4. Similarity Computation Details:")  # 打印阶段标题
    logits_per_image = outputs.logits_per_image  # 图像对各文本的相似度得分，类型 Tensor，形状 (1,3)，值=logit_scale × 余弦相似度
    logits_per_text = outputs.logits_per_text    # 文本对图像的相似度得分，类型 Tensor，形状 (3,1)，为 logits_per_image 的转置

    print(f"   Image-to-text logits: {logits_per_image.shape}")  # 打印图像→文本方向的logits形状：(num_images, num_texts)
    print(f"   Text-to-image logits: {logits_per_text.shape}")  # 打印文本→图像方向的logits形状：(num_texts, num_images)
    # logit_scale 是可学习温度参数，exp后约为100，放大余弦相似度使分布更尖锐，便于分类
    print(f"   Logit scale parameter: {model.logit_scale.exp().item():.4f}")  # 打印温度参数exp值（标量），越大分布越尖锐

    # 对图像→文本方向的logits做softmax（dim=1），得到每张图像匹配各文本的概率
    probs_image_to_text = logits_per_image.softmax(dim=1)  # 沿文本维度softmax，返回 Tensor，形状 (1, 3)，每行3个概率值之和为1
    # 对文本→图像方向的logits做softmax（dim=0），得到每段文本匹配各图像的概率
    probs_text_to_image = logits_per_text.softmax(dim=0)  # 沿图像维度softmax，返回 Tensor，形状 (3, 1)，每列各概率值之和为1

    print(f"   Image-to-text probabilities shape: {probs_image_to_text.shape}")  # 打印图→文概率张量形状：(num_images, num_texts)
    print(f"   Text-to-image probabilities shape: {probs_text_to_image.shape}")  # 打印文→图概率张量形状：(num_texts, num_images)
    print()  # 空行分隔

    # ============ 阶段5：详细结果展示 ============
    print("5. Detailed Results:")  # 打印阶段标题
    print('-'*30)  # 打印分隔线

    print("Raw similarity scores (logits):")  # 提示打印原始相似度得分（softmax之前的原始logit值）
    for i, text in enumerate(texts):  # 遍历每段文本（i为索引，text为文本内容），打印对应的原始logit值
        print(f"   '{text}': {logits_per_image[0, i].item():.4f}")  # 取第0张图像对第i段文本的得分，.item()将标量张量转为Python浮点数
    print()  # 空行分隔

    print("Probability distribution:")  # 提示打印softmax概率分布
    probs = probs_image_to_text[0]  # 取第一张（也是唯一一张）图像的概率向量，类型 Tensor，形状 (3,)，3个元素对应3段文本
    for i, text in enumerate(texts):  # 遍历每段文本，打印其匹配概率和百分比形式
        print(f"   '{text}': {probs[i].item():.4f} ({probs[i].item()*100:.2f}%)")  # 打印概率值（4位小数）和百分比（2位小数）
    print()  # 空行分隔

    best_match_idx = torch.argmax(probs).item()  # 找到概率最高的文本的索引（int），对应最匹配的候选文本
    # 打印最终预测结果：最匹配的文本描述及其对应概率
    print(f"Best match: '{texts[best_match_idx]}' with probability {probs[best_match_idx].item():.4f}")  # 打印最佳匹配文本及其概率
    print()  # 空行分隔

# ============ 阶段6：模型参数统计 ============
print("6. Model Parameter Statistics:")  # 打印阶段标题
print('-'*40)  # 打印分隔线
total_params = sum(p.numel() for p in model.parameters())  # 统计模型全部参数量（int），p.numel()返回每个参数张量的元素总数
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # 统计可训练参数量（int），requires_grad=True表示该参数参与梯度更新

print(f"Total parameters: {total_params:,}")  # 打印总参数量，使用逗号格式化（如 151,277,313）便于阅读
print(f"Trainable parameters: {trainable_params:,}")  # 打印可训练参数量，与总量相同时说明所有参数均参与训练

# 分别统计各子模块的参数量，用于了解模型各部分的参数分布
text_params = sum(p.numel() for p in model.text_model.parameters())  # 文本Transformer（12层）总参数量（int）
vision_params = sum(p.numel() for p in model.vision_model.parameters())  # 视觉Transformer/ViT（12层）总参数量（int）
text_proj_params = sum(p.numel() for p in model.text_projection.parameters())  # 文本投影头（Linear层）参数量（int），实现512→512映射
visual_proj_params = sum(p.numel() for p in model.visual_projection.parameters())  # 视觉投影头（Linear层）参数量（int），实现768→512映射

print(f"Text model parameters: {text_params:,}")  # 打印文本模型参数量
print(f"Vision model parameters: {vision_params:,}")  # 打印视觉模型参数量
print(f"Text projection parameters: {text_proj_params:,}")  # 打印文本投影层参数量（512×512=262,144）
print(f"Visual projection parameters: {visual_proj_params:,}")  # 打印视觉投影层参数量（768×512=393,216）
print(f"Logit scale parameter: 1")  # logit_scale 是单个可学习标量（Tensor，shape=()），参数量为1

print('='*100)  # 打印结束分隔线
print("Analysis Complete!")  # 打印完成提示

CLIP Model Architecture:
CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (la

## 三、手动实现 CLIP 模型

本节从零手动实现简化版 CLIP 模型，加深对对比学习原理的理解。
本节引入完整 ViT 图像编码器（PatchEmbedding + MultiHeadAttention），贴近真实实现。

> **关键知识点**：相似度矩阵在 batch 内计算，论文中的 N 就是 batch size，
> 对角线元素为正样本对（图像 i 与文本 i），其余为负样本对。

### 3.1 带完整 ViT 图像编码器的 CLIP 实现

使用 `PatchEmbedding + MultiHeadAttention + ImageEncoder` 手动搭建完整 ViT 视觉编码器，
同时将文本编码器升级为带自注意力机制的简单 Transformer，进一步贴近真实 CLIP 模型结构。

In [ ]:
import math  # 导入数学库，用于 sqrt 等运算
import torch  # 导入PyTorch，提供张量和自动求导
import torch.nn as nn  # 导入神经网络模块，提供 Linear、LayerNorm 等组件
import torch.nn.functional as F  # 导入函数式接口，提供 softmax、cross_entropy 等


# ============================================================
# 一、图像分块嵌入层（Patch Embedding）
# ============================================================
class PatchEmbedding(nn.Module):
    """
    图像分块嵌入层（Patch Embedding）。

    将输入图像切分为固定大小的 patch，通过卷积层将每个 patch 线性映射为嵌入向量，
    在序列开头拼接可学习的 [CLS] token，并加上可学习的位置编码。
    对应 CLIP/ViT 论文中"将图像转化为 token 序列"的核心步骤。
    """

    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=768):
        """
        初始化图像分块嵌入层。

        参数：
            image_size  (int): 输入图像高宽，默认 224（即 224×224 像素）
            patch_size  (int): 每个 patch 的高宽，默认 16（即 16×16 像素）
            in_channels (int): 输入图像通道数，默认 3（RGB）
            embed_dim   (int): patch 映射后的嵌入维度，默认 768（ViT-B/16 配置）
        """
        super().__init__()
        # patch 总数 = (image_size / patch_size)^2，例如 (224/16)^2 = 14^2 = 196
        self.num_patches = (image_size // patch_size) ** 2

        # 卷积层实现"分块+线性映射"：kernel_size=stride=patch_size 保证无重叠切分
        # 输入 (B,3,224,224) → 输出 (B,embed_dim,14,14)
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

        # 可学习的 [CLS] token，形状 (1,1,embed_dim)；扩展到批次后拼接到序列开头，作为全图全局表示
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        # 可学习的位置编码，形状 (1, num_patches+1, embed_dim)；+1 对应 [CLS] token 的位置
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))

    def forward(self, x):
        """
        前向传播：图像 → 带位置编码的 patch token 序列。

        参数：
            x (torch.Tensor): 输入图像，形状 (B, in_channels, image_size, image_size)
                              B=批次大小, in_channels=3, image_size=224

        返回：
            torch.Tensor: 形状 (B, num_patches+1, embed_dim)
                          - B           : 批次大小
                          - num_patches+1: 196 个 patch token + 1 个 [CLS] token = 197
                          - embed_dim   : 每个 token 的嵌入维度（768）
        """
        B = x.shape[0]  # 批次大小，类型 int

        # 卷积分块并展平：(B,3,224,224) → (B,768,14,14) → flatten(2) → (B,768,196)
        #                                                  → transpose(1,2) → (B,196,768)
        x = self.projection(x).flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)

        # 将 [CLS] token 复制到每个样本：(1,1,768) → expand → (B,1,768)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        # 在序列维度拼接 [CLS] token：(B,1,768) + (B,196,768) → (B,197,768)
        x = torch.cat((cls_tokens, x), dim=1)

        # 加位置编码（形状 (1,197,768) 自动广播），为每个 token 引入位置信息
        x = x + self.pos_embed
        return x  # (B, num_patches+1, embed_dim)


# ============================================================
# 二、多头自注意力（Multi-Head Self-Attention）
# ============================================================
class MultiHeadAttention(nn.Module):
    """
    多头自注意力机制（Multi-Head Self-Attention）。

    将输入通过联合 QKV 投影后分头，并行计算缩放点积注意力，
    拼接各头结果后通过输出投影恢复原始维度。
    是 Transformer 编码器的核心计算单元。
    """

    def __init__(self, embed_dim, num_heads):
        """
        参数：
            embed_dim (int): 输入特征维度（每个 token 的嵌入维度）
            num_heads (int): 注意力头数，embed_dim 必须能被 num_heads 整除
                             每头维度 head_dim = embed_dim // num_heads
        """
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim 必须能被 num_heads 整除"
        self.num_heads = num_heads  # 注意力头数
        self.head_dim = embed_dim // num_heads  # 每头的特征维度
        # 缩放因子 1/√head_dim，防止点积值过大使 softmax 进入梯度消失区域
        self.scale = self.head_dim ** -0.5

        # 联合 QKV 投影：一次线性变换同时得到 Q、K、V，输出维度为 embed_dim×3
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        # 多头拼接后的输出投影，将 embed_dim 映射回 embed_dim
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        """
        参数：
            x (torch.Tensor): 形状 (B, N, embed_dim)
                              B=批次, N=序列长度(num_patches+1), embed_dim=特征维度

        返回：
            torch.Tensor: 形状 (B, N, embed_dim)，与输入形状相同
        """
        B, N, C = x.shape  # B=批次, N=序列长度, C=embed_dim

        # QKV 投影并分头
        # (B,N,C) → Linear → (B,N,3C) → reshape → (B,N,3,heads,head_dim)
        # → permute(2,0,3,1,4) → (3, B, heads, N, head_dim)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # 各形状 (B, num_heads, N, head_dim)

        # 缩放点积注意力：(B,heads,N,head_dim) @ (B,heads,head_dim,N) → (B,heads,N,N)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # 注意力原始分数
        attn = attn.softmax(dim=-1)  # 沿 key 维度 softmax，得注意力权重，形状 (B,heads,N,N)

        # 加权聚合：(B,heads,N,N) @ (B,heads,N,head_dim) → (B,heads,N,head_dim)
        # transpose(1,2) → (B,N,heads,head_dim) → reshape → (B,N,C)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(x)  # 输出投影，形状 (B, N, embed_dim)


# ============================================================
# 三、Transformer 编码器块（Pre-Norm 风格）
# ============================================================
class TransformerBlock(nn.Module):
    """
    单个 Transformer 编码器块（Pre-Norm 风格）。

    执行顺序（Pre-Norm）：
        LayerNorm → MultiHeadAttention → 残差连接
        LayerNorm → MLP（升维→GELU→降维）→ 残差连接

    与 Post-Norm（BERT 风格）相比，Pre-Norm 训练更稳定，深层网络收敛更快。
    """

    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0):
        """
        参数：
            embed_dim  (int)  : 特征维度
            num_heads  (int)  : 多头注意力的头数
            mlp_ratio  (float): MLP 隐藏层相对于 embed_dim 的扩张倍数，默认 4.0
        """
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)  # Attention 前的 LayerNorm
        self.attn = MultiHeadAttention(embed_dim, num_heads)  # 多头自注意力
        self.norm2 = nn.LayerNorm(embed_dim)  # MLP 前的 LayerNorm
        mlp_hidden = int(embed_dim * mlp_ratio)  # MLP 隐藏层维度，默认为 embed_dim×4
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),  # 升维：embed_dim → mlp_hidden
            nn.GELU(),                          # GELU 激活（比 ReLU 更平滑）
            nn.Linear(mlp_hidden, embed_dim),  # 降维：mlp_hidden → embed_dim
        )

    def forward(self, x):
        """
        参数：
            x (torch.Tensor): 形状 (B, N, embed_dim)

        返回：
            torch.Tensor: 形状 (B, N, embed_dim)，与输入相同
        """
        # Pre-Norm Attention + 残差：先归一化，再注意力，最后加回原始输入
        x = x + self.attn(self.norm1(x))
        # Pre-Norm MLP + 残差：先归一化，再 MLP，最后加回中间结果
        x = x + self.mlp(self.norm2(x))
        return x


# ============================================================
# 四、ViT 图像编码器
# ============================================================
class ImageEncoder(nn.Module):
    """
    基于 ViT 的图像编码器（Vision Transformer Image Encoder）。

    完整流程：
        输入图像
        → PatchEmbedding（分块+位置编码）
        → num_layers 个 TransformerBlock
        → LayerNorm
        → 取 [CLS] token（索引 0）作为整图全局表示
        → 输出形状 (B, embed_dim)
    """

    def __init__(self, embed_dim=768, image_size=224, patch_size=16,
                 num_layers=12, num_heads=12, mlp_ratio=4.0):
        """
        参数：
            embed_dim  (int)  : token 嵌入维度，也是最终输出的图像特征维度，默认 768
            image_size (int)  : 输入图像高宽，默认 224
            patch_size (int)  : patch 高宽，默认 16
            num_layers (int)  : Transformer 块的层数，默认 12
            num_heads  (int)  : 多头注意力的头数，默认 12
            mlp_ratio  (float): MLP 扩张倍数，默认 4.0
        """
        super().__init__()
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, embed_dim)
        # ModuleList 存储各 TransformerBlock，确保参数被 PyTorch 正确注册和优化
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)  # 最终归一化层，对应 CLIP ViT 的 post_layernorm

    def forward(self, x):
        """
        参数：
            x (torch.Tensor): 输入图像，形状 (B, 3, image_size, image_size)
                              B=批次大小, 3=RGB通道, image_size=224

        返回：
            torch.Tensor: 图像全局特征，形状 (B, embed_dim)
                          取 [CLS] token（序列索引 0）的输出，聚合了全部 patch 的信息
        """
        x = self.patch_embed(x)   # (B, num_patches+1, embed_dim)，即 (B,197,768)
        for block in self.blocks:
            x = block(x)           # 每层 TransformerBlock，形状保持 (B, 197, 768)
        x = self.norm(x)           # 最终 LayerNorm，形状仍为 (B, 197, 768)
        return x[:, 0]             # 取索引0即 [CLS] token，返回形状 (B, embed_dim)


# ============================================================
# 五、简化版文本编码器（单层 Transformer + 均值池化）
# ============================================================
class TextEncoder(nn.Module):
    """
    简化版文本编码器（教学用途）。

    完整流程：
        token IDs (B, seq_len)
        → Token Embedding + 位置编码 → (B, seq_len, text_dim)
        → 1 个 TransformerBlock
        → LayerNorm
        → 均值池化（对 seq_len 维取均值）→ (B, text_dim)
        → 线性投影到共享嵌入空间 → (B, embed_dim)

    教学简化说明：
        - 真实 CLIP 使用 12 层 Transformer，此处用 1 层
        - 真实 CLIP 取 [EOS] token，此处使用均值池化
        - 真实 CLIP 使用 BPE Tokenizer，此处使用随机 Embedding 演示
    """

    def __init__(self, vocab_size, text_dim, embed_dim, max_seq_len=32):
        """
        参数：
            vocab_size  (int): 词表大小（token 种类数）
            text_dim    (int): token 嵌入维度（文本编码器内部维度）
            embed_dim   (int): 输出嵌入维度，与图像编码器对齐，用于余弦相似度计算
            max_seq_len (int): 最大序列长度，默认 32
        """
        super().__init__()
        # Token 嵌入层：将整数 token ID 映射为 text_dim 维向量
        self.token_embed = nn.Embedding(vocab_size, text_dim)
        # 可学习位置编码，形状 (1, max_seq_len, text_dim)，在 __init__ 中定义确保参数持久
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, text_dim))
        # 单层 Transformer 块（教学简化，真实 CLIP 用12层）
        self.block = TransformerBlock(text_dim, num_heads=8)
        self.norm = nn.LayerNorm(text_dim)  # 最终归一化
        # 投影层：将 text_dim 维向量映射到与图像编码器对齐的 embed_dim 维共享嵌入空间
        self.proj = nn.Linear(text_dim, embed_dim)

    def forward(self, token_ids):
        """
        参数：
            token_ids (torch.LongTensor): token ID 序列，形状 (B, seq_len)
                                          B=批次大小, seq_len=序列长度（≤ max_seq_len）

        返回：
            torch.Tensor: 文本嵌入向量，形状 (B, embed_dim)
                          每条文本对应一个 embed_dim 维的全局表示向量
        """
        seq_len = token_ids.size(1)  # 当前序列长度（≤ max_seq_len），类型 int

        # Token 嵌入 + 位置编码，位置编码截取前 seq_len 个位置以支持动态长度
        x = self.token_embed(token_ids) + self.pos_embed[:, :seq_len]  # (B, seq_len, text_dim)

        x = self.block(x)   # 单层 Transformer 编码，形状保持 (B, seq_len, text_dim)
        x = self.norm(x)    # LayerNorm，形状保持 (B, seq_len, text_dim)

        # 均值池化：对 seq_len 维取均值，将变长序列压缩为固定维度的句子级向量
        x = x.mean(dim=1)   # (B, text_dim)

        return self.proj(x)  # 线性投影到共享嵌入空间，返回形状 (B, embed_dim)


# ============================================================
# 六、CLIP 完整模型
# ============================================================
class CLIP(nn.Module):
    """
    简化版 CLIP 对比学习模型（双塔结构）。

    双塔结构：
        图像塔：ViT 图像编码器（PatchEmbedding + 多层 TransformerBlock）
        文本塔：简化 Transformer 文本编码器（Embedding + 1层 TransformerBlock + 均值池化）

    训练目标：通过对比学习，使同语义图文对的嵌入在共享空间中余弦相似度最大化，
              不同语义图文对的余弦相似度最小化（InfoNCE 损失）。
    """

    def __init__(self, embed_dim=512, vocab_size=1000, text_dim=256):
        """
        参数：
            embed_dim  (int): 图文共享嵌入空间维度，默认 512
            vocab_size (int): 文本词表大小，默认 1000（教学简化，真实CLIP词表约49408）
            text_dim   (int): 文本编码器内部维度，默认 256
        """
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim=embed_dim)
        self.text_encoder = TextEncoder(
            vocab_size=vocab_size, text_dim=text_dim, embed_dim=embed_dim
        )
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)
        # logit_scale：可学习温度参数（标量），类型 nn.Parameter(Tensor, shape=[])，参与梯度更新
        # 初始值 2.6592，前向传播取指数：exp(2.6592) ≈ 14.3；训练后约为 exp(4.605) ≈ 100
        # 作用：缩放余弦相似度矩阵，logit_scale 越大 → 分布越尖锐 → 正样本区分越自信
        # CLIP 论文对其上限裁剪到 ln(100)，防止训练不稳定

    def forward(self, images, token_ids):
        """
        前向传播：计算图像与文本的跨模态相似度矩阵。

        参数：
            images    (torch.Tensor)    : 图像张量，形状 (B, 3, 224, 224)
                                          B=批次大小（即论文中的N），3=RGB通道
            token_ids (torch.LongTensor): 文本 token ID 序列，形状 (B, seq_len)
                                          B 与 images 相同，seq_len=序列长度

        返回：
            tuple(torch.Tensor, torch.Tensor):
                logits_per_image: 形状 (B, B)，第[i,j]元素=第i张图与第j段文本的相似度×温度
                                  对角线为正样本对，其余为负样本对
                logits_per_text : 形状 (B, B)，为 logits_per_image 的转置
        """
        # 各自编码：图像 → (B, embed_dim)，文本 → (B, embed_dim)
        image_features = self.image_encoder(images)       # (B, embed_dim)
        text_features = self.text_encoder(token_ids)      # (B, embed_dim)

        # L2 归一化（p=2），使各特征向量模长=1，点积等价于余弦相似度
        image_features = F.normalize(image_features, p=2, dim=-1)  # (B, embed_dim)
        text_features = F.normalize(text_features, p=2, dim=-1)    # (B, embed_dim)

        # 温度缩放相似度矩阵
        logit_scale = self.logit_scale.exp()  # 标量，约 14.3（初始）或 100（训练后）
        # (B,embed_dim) @ (embed_dim,B) = (B,B)；第[i,j]元素=第i图与第j文本的余弦相似度×logit_scale
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()  # 转置，形状 (B, B)

        return logits_per_image, logits_per_text


# ============================================================
# 七、对比损失函数（对称 InfoNCE Loss）
# ============================================================
def clip_loss(logits_per_image, logits_per_text):
    """
    计算 CLIP 对称 InfoNCE 损失。

    对图像→文本和文本→图像两个方向分别计算交叉熵损失，取均值作为总损失。
    标签为对角线索引 [0,1,...,N-1]：第 i 张图像与第 i 段文本互为正样本对。

    参数：
        logits_per_image (torch.Tensor): 形状 (B, B)，图像对所有文本的相似度得分
                                         第i行第j列：第i张图与第j段文本的得分
        logits_per_text  (torch.Tensor): 形状 (B, B)，logits_per_image 的转置

    返回：
        torch.Tensor: 标量，CLIP 对比损失值（两方向交叉熵的均值）
    """
    B = logits_per_image.size(0)  # 批次大小，类型 int
    # 标签 [0,1,...,B-1]：第 i 张图的正样本是第 i 段文本（对角线位置），形状 (B,)，类型 long
    labels = torch.arange(B, device=logits_per_image.device)

    loss_i2t = F.cross_entropy(logits_per_image, labels)  # 图像→文本方向，标量 Tensor
    loss_t2i = F.cross_entropy(logits_per_text, labels)   # 文本→图像方向，标量 Tensor
    return (loss_i2t + loss_t2i) / 2  # 对称损失取均值，返回标量 Tensor


# ============================================================
# 八、运行示例
# ============================================================
batch_size = 4    # 批次大小（论文中的 N），类型 int
embed_dim = 512   # 图文共享嵌入空间维度，类型 int
vocab_size = 1000  # 词表大小（教学简化），类型 int
text_dim = 256    # 文本编码器内部维度，类型 int
seq_len = 32      # 文本序列长度，类型 int

# 随机生成图像张量，形状 (B,3,224,224)，替代真实图像数据
images = torch.randn(batch_size, 3, 224, 224)
# 随机生成 token ID 序列，形状 (B,seq_len)，整数范围 [0, vocab_size)，替代真实 Tokenizer 输出
token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

# 初始化 CLIP 模型
model = CLIP(embed_dim=embed_dim, vocab_size=vocab_size, text_dim=text_dim)

# 前向传播：得到图文相似度矩阵
logits_per_image, logits_per_text = model(images, token_ids)
# 计算对称 InfoNCE 损失
loss = clip_loss(logits_per_image, logits_per_text)

print("logits_per_image shape:", logits_per_image.shape)  # 期望 torch.Size([4, 4])：4张图 × 4段文本
print("logits_per_text  shape:", logits_per_text.shape)   # 期望 torch.Size([4, 4])：4段文本 × 4张图
print("Loss:", loss.item())  # 随机初始化时约为 ln(4) ≈ 1.386（均匀分布时的理论值）


## 四、CLIP 架构特性：双塔结构与无交叉注意力

### 4.1 CLIP 有交叉注意力（Cross-Attention）吗？

**结论：CLIP 没有交叉注意力。**

从前面打印的模型结构可以直接验证这一点：

```
CLIPModel(
  (text_model): CLIPTextTransformer
    (encoder): CLIPEncoder
      (0-11): 12 x CLIPEncoderLayer
        (self_attn): CLIPSdpaAttention   ← 只有自注意力，无交叉注意力

  (vision_model): CLIPVisionTransformer
    (encoder): CLIPEncoder
      (0-11): 12 x CLIPEncoderLayer
        (self_attn): CLIPSdpaAttention   ← 只有自注意力，无交叉注意力
)
```

每一层只有 `self_attn`（自注意力），文本编码器和图像编码器**完全独立**，在各自内部只做模态内部的自注意力，两个编码器之间没有任何交叉注意力模块。

---

### 4.2 CLIP 的图文交互方式：双塔结构

CLIP 的图文"交互"不发生在编码器内部，而是发生在**编码器之外**，通过以下方式实现：

```
图像 (224×224×3)
    → ViT 图像编码器（12 层自注意力，内部只看图像）
    → visual_projection（768 → 512）
    → L2 归一化
    → image_embeds  (N, 512)
                            ↘
                         矩阵点积 → 相似度矩阵 (N, M)
                            ↗
    → L2 归一化
    → text_projection（512 → 512）
    → Text Transformer（12 层自注意力，内部只看文本）
文本 (token序列)
```

**核心流程：**

1. 图像和文本各自独立地通过自己的 Transformer 编码器（模态内自注意力）
2. 各自经过投影层映射到同一维度（512 维共享嵌入空间）
3. 做 L2 归一化，使每个向量长度为 1（余弦相似度 = 点积）
4. 通过矩阵乘法计算图文余弦相似度矩阵
5. 乘以温度参数 `logit_scale`，得到最终 logits

这种架构称为**双塔结构（Dual Encoder / Two-Tower）**，两塔独立编码，只在最后的嵌入空间中"碰面"。

### 4.3 双塔结构的优缺点

| 维度 | 说明 |
|---|---|
| **优点：推理速度快** | 图像和文本的特征可以分别**提前计算并缓存**，检索时只需做向量点积，适合大规模图文检索（如以图搜图） |
| **优点：可扩展性强** | 4 亿图文对的大规模对比预训练成为可能，训练效率高 |
| **缺点：细粒度理解弱** | 由于两个模态在编码过程中完全独立，模型无法在内部进行细粒度的图文对齐（如"图中左边的红色物体"这类局部关联） |
| **缺点：交互发生太晚** | 图文交互只发生在最后的单一向量之间，丢失了序列级别的位置和局部信息 |

---

### 4.4 与 ALBEF 的架构对比

| 特性 | CLIP | ALBEF |
|---|---|---|
| **整体架构** | 双塔（Dual Encoder） | 单流融合（Multimodal Encoder） |
| **图文交互位置** | 编码器外（相似度矩阵，最后一步） | 编码器内（多模态编码器中的交叉注意力层） |
| **交叉注意力** | ❌ 无 | ✅ 有（文本 Query 注意到图像 Key/Value） |
| **图文对齐粒度** | 粗粒度（整体向量对齐） | 细粒度（token 级别对齐） |
| **推理速度** | 快（特征可预计算） | 慢（每次推理需联合编码） |
| **细粒度理解能力** | 较弱 | 较强（如 VQA、图文推理） |
| **适合任务** | 大规模零样本图文检索/分类 | VQA、图像描述、图文推理等需要细粒度理解的任务 |

**关键区别示意：**

```
CLIP（双塔，无交叉注意力）：
  [Image Encoder] ──→ image_vec ──┐
                                    ├── dot product → score
  [Text  Encoder] ──→  text_vec ──┘

ALBEF（融合，有交叉注意力）：
  [Image Encoder] ──→ image_tokens ──→ 作为 Key/Value ──┐
                                                           ├── Cross-Attention → 融合特征
  [Text  Encoder] ──→  text_tokens ──→  作为 Query    ──┘
```

> **总结**：CLIP 通过双塔结构和对比学习实现了高效的大规模图文预训练，但代价是放弃了细粒度的跨模态交互。ALBEF 等后续工作正是为了弥补这一缺陷，在两个独立编码器之后引入带交叉注意力的多模态融合编码器，兼顾检索效率与理解能力。